 # SEC EDGAR Data Processing: Deriving Clean Quarterly Data
 This notebook demonstrates how to load raw SEC EDGAR XBRL data from a Parquet file
 and process it to extract clean, discrete quarterly numbers.

 A common challenge with SEC data is that Q4 numbers are rarely reported explicitly
 as a discrete 3-month value (`qtrs=1`) in the 10-K filings. Instead, the 10-K primarily
 provides the annual total (`qtrs=4`).

In [ ]:
import pandas as pd
import plotly.express as px

# Define the path to your parquet file
parquet_path = 'data/TSLA_nums.parquet'

# Read the Parquet file into a Pandas DataFrame
df = pd.read_parquet(parquet_path)

 ## 1. Filtering and Data Prep
 First, we filter the raw dataset for the specific accounting tag we want (`GrossProfit`)
 and convert the period end dates (`ddate`) to standard datetime objects.
 We also filter for `dimn == 0` which represents the "Consolidated" company total,
 ensuring we aren't accidentally pulling segmented business unit data.

In [ ]:
# Filter for 'GrossProfit' and prepare dates
gross_profit_df = df[df['tag'] == 'GrossProfit'].copy()
gross_profit_df['ddate'] = pd.to_datetime(gross_profit_df['ddate'])

# Filter for Consolidated Totals Only (ALL Years)
plot_df = gross_profit_df[gross_profit_df['dimn'] == 0].copy()

# Sort and drop duplicates to ensure we get the first/latest reported numbers if there are restatements
plot_df = plot_df.sort_values(['ddate', 'adsh']).drop_duplicates(subset=['ddate', 'qtrs'], keep='first')

 ## 2. Isolating Q1, Q2, and Q3
 Quarters 1, 2, and 3 are typically reported in 10-Q filings with a `qtrs` value of 1
 (meaning it represents exactly 1 quarter / 3 months of duration). We can extract these directly.

In [ ]:
# Separate Q1, Q2, Q3 (where qtrs == 1 explicitly exists)
clean_df = plot_df[plot_df['qtrs'] == 1].copy()

 ## 3. Deriving Q4 Data
 **How to get precise Q4 Data:**
 Companies do not typically file a standard 10-Q for the 4th quarter. Instead, they file an annual 10-K.
 Because of this, the `qtrs=1` tag for the period ending on Q4's date (often Dec 31) is usually missing.

 To systematically calculate the discrete Q4 value, we use the formula:
 **`Q4 Value = Annual Year-to-Date Value (qtrs=4) - Q3 Year-to-Date Value (qtrs=3)`**

 The loop below matches the `qtrs=3` row with the `qtrs=4` row for the exact same fiscal year
 and calculates the discrete Q4 difference.

In [ ]:
# Find all unique years in our dataset
years = plot_df['ddate'].dt.year.unique()
q4_rows = []

for year in years:
    # Find Annual data (qtrs=4) and Q3 YTD data (qtrs=3) for the current year
    annual_data = plot_df[(plot_df['ddate'].dt.year == year) & (plot_df['qtrs'] == 4)]
    q3_ytd_data = plot_df[(plot_df['ddate'].dt.year == year) & (plot_df['qtrs'] == 3)]
    
    # If both exist for this given year, we can systematically calculate Q4
    if not annual_data.empty and not q3_ytd_data.empty:
        annual_val = annual_data['value'].iloc[0]
        q3_ytd_val = q3_ytd_data['value'].iloc[0]
        
        # The date for Q4 is the exact same as the annual report end date
        annual_date = annual_data['ddate'].iloc[0] 
        
        q4_val = annual_val - q3_ytd_val
        
        # Store the calculated Q4 row as a derived qtrs=1 record
        q4_rows.append({
            'ddate': annual_date,
            'value': q4_val,
            'qtrs': 1,
            'dimn': 0,
            'note': 'Derived Q4 (Annual - Q3 YTD)' # Helpful metadata tag for auditing
        })

# Append all the newly calculated Q4 rows back to our clean dataframe
if q4_rows:
    q4_df = pd.DataFrame(q4_rows)
    clean_df = pd.concat([clean_df, q4_df], ignore_index=True)

# Ensure the final dataframe is sorted chronologically
clean_df = clean_df.sort_values('ddate').reset_index(drop=True)

print("Cleaned Data Preview (Includes Derived Q4 data):")
print(clean_df[['ddate', 'value', 'qtrs', 'note'] if 'note' in clean_df.columns else ['ddate', 'value', 'qtrs']].tail(8))

Cleaned Data Preview (Includes Derived Q4 data):
        ddate         value  qtrs                          note
65 2024-03-31  3.696000e+09     1                           NaN
66 2024-06-30  4.578000e+09     1                           NaN
67 2024-09-30  4.997000e+09     1                           NaN
68 2024-12-31  4.179000e+09     1  Derived Q4 (Annual - Q3 YTD)
69 2025-03-31  3.153000e+09     1                           NaN
70 2025-06-30  3.878000e+09     1                           NaN
71 2025-09-30  5.054000e+09     1                           NaN
72 2025-12-31  5.009000e+09     1  Derived Q4 (Annual - Q3 YTD)


 ## 4. Visualizing the Continuous Quarterly Series
 Now that we have a contiguous time series of purely discrete, 3-month quarterly values (`qtrs=1`),
 we can visualize it cleanly without overlapping or compounding Annual/YTD spikes.

In [ ]:
# Plot the final, clean discrete quarterly data for ALL time
fig = px.line(
    clean_df,
    x='ddate',
    y='value',
    title='Tesla Consolidated Quarterly Gross Profit (All Time) – Adjusted for Q4',
    labels={'ddate': 'Period End Date', 'value': 'Quarterly Gross Profit (USD)'},
    markers=True,
    template='plotly_white'
)
fig.show()